<a href="https://colab.research.google.com/github/AureliaVDB/TickIt_Data_Pipeline/blob/main/TickIt_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark -q

In [56]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import IntegerType, TimestampType, BooleanType, DataType, DecimalType
import os, glob, shutil
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import os

In [3]:
spark = (SparkSession.builder
         .appName("TickIt_Medallion_Pipeline")
         .getOrCreate()
         )

In [4]:
from google.colab import drive

drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/TickIt_Project"

RAW = f"{BASE}/raw"  # csv files
BRONZE = f"{BASE}/bronze"  # storage for bronze layer
SILVER = f"{BASE}/silver"  # storage for silver layer
GOLD = f"{BASE}/gold"  # storage for gold layer

for path in [RAW, BRONZE, SILVER, GOLD]:
    os.makedirs(path, exist_ok=True)

Mounted at /content/drive


Bronze Layer

In [5]:
# list of all the source files
raw_files = []
for f in os.listdir(RAW):
    if f.endswith(".csv"):
        raw_files.append(f)

print(f"files in list: {raw_files}")

files in list: ['orders.csv', 'payments.csv', 'events.csv', 'event_categories.csv', 'ticket_types.csv', 'customers.csv', 'reviews.csv']


In [6]:
for file in raw_files:
  input_path = os.path.join(RAW, file)
  table_name = os.path.splitext(file)[0]
  output_path = os.path.join(BRONZE, table_name)

  df = spark.read.csv(input_path, header=True) # load the raw data
  df.write.mode("overwrite").parquet(output_path) # write to bronze layer

  print(f"{table_name.upper()}")
  print()
  print("Schema")
  df.printSchema()
  print(f"Row Count: {df.count()}")
  print()
  print("Sample Records")
  df.show(5)
  print()

ORDERS

Schema
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- ticket_type_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- event_date: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- promo_code: string (nullable = true)

Row Count: 52520

Sample Records
+--------------+----------------+--------+--------------+-------------------+----------+--------+------------+----------+
|      order_id|     customer_id|event_id|ticket_type_id|         order_date|event_date|quantity|order_status|promo_code|
+--------------+----------------+--------+--------------+-------------------+----------+--------+------------+----------+
|    ORD0039854|b2744cf668394fac| EVT0050|       TT00106|2022-08-15 10:00:00|2022-09-03|       2|   completed|      NULL|
|    ORD0002676|ad9181b36ee84121| EVT0092|       TT00194|2021-04-15 23:00:00|20

Silver Layer

In [7]:
bronze_tables = {}

for folder_name in os.listdir(BRONZE):
  folder_path = os.path.join(BRONZE, folder_name)
  if os.path.isdir(folder_path):
    bronze_tables[folder_name] = spark.read.parquet(folder_path)


In [8]:
#Orders

orders_raw = bronze_tables["orders"]

orders_raw.printSchema()

orders_raw.show(5)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- ticket_type_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- event_date: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- promo_code: string (nullable = true)

+--------------+----------------+--------+--------------+-------------------+----------+--------+------------+----------+
|      order_id|     customer_id|event_id|ticket_type_id|         order_date|event_date|quantity|order_status|promo_code|
+--------------+----------------+--------+--------------+-------------------+----------+--------+------------+----------+
|    ORD0039854|b2744cf668394fac| EVT0050|       TT00106|2022-08-15 10:00:00|2022-09-03|       2|   completed|      NULL|
|    ORD0002676|ad9181b36ee84121| EVT0092|       TT00194|2021-04-15 23:00:00|2021-06-30|       6|   completed|      NULL|
|    

In [9]:
# fixing variable type
orders_cleaned = (
    orders_raw.withColumn("quantity", F.col("quantity").cast(IntegerType()))
    .withColumn("order_date", F.to_timestamp(F.col("order_date")))
    .withColumn("event_date", F.to_date(F.col("event_date")))
)

orders_cleaned.printSchema()
orders_cleaned.show(5, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- ticket_type_id: string (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- event_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- order_status: string (nullable = true)
 |-- promo_code: string (nullable = true)

+--------------+----------------+--------+--------------+-------------------+----------+--------+------------+----------+
|order_id      |customer_id     |event_id|ticket_type_id|order_date         |event_date|quantity|order_status|promo_code|
+--------------+----------------+--------+--------------+-------------------+----------+--------+------------+----------+
|ORD0039854    |b2744cf668394fac|EVT0050 |TT00106       |2022-08-15 10:00:00|2022-09-03|2       |completed   |NULL      |
|ORD0002676    |ad9181b36ee84121|EVT0092 |TT00194       |2021-04-15 23:00:00|2021-06-30|6       |completed   |NULL      |
|OR

In [10]:
# nullable

orders_cleaned.select(
    [
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in [
            "order_id",
            "customer_id",
            "event_id",
            "ticket_type_id",
            "order_date",
            "quantity",
            "promo_code",
        ]
    ]
).show()

+--------+-----------+--------+--------------+----------+--------+----------+
|order_id|customer_id|event_id|ticket_type_id|order_date|quantity|promo_code|
+--------+-----------+--------+--------------+----------+--------+----------+
|       0|        270|       0|             0|         0|       0|     29352|
+--------+-----------+--------+--------------+----------+--------+----------+



In [11]:

missing_customers = orders_cleaned.filter(F.col("customer_id").isNull())

print(f"Total rows with missing customer_id: {missing_customers.count()}")
missing_customers.show(10, truncate=False)


Total rows with missing customer_id: 270
+----------+-----------+--------+--------------+-------------------+----------+--------+------------+----------+
|order_id  |customer_id|event_id|ticket_type_id|order_date         |event_date|quantity|order_status|promo_code|
+----------+-----------+--------+--------------+-------------------+----------+--------+------------+----------+
|ORD0037574|NULL       |EVT0181 |TT00366       |2022-04-12 06:00:00|2022-04-20|4       |completed   |NULL      |
|ORD0003884|NULL       |EVT0340 |TT00705       |2022-10-17 19:00:00|2022-11-13|4       |completed   |STUDENT15 |
|ORD0037061|NULL       |EVT0408 |TT00835       |2025-04-18 08:00:00|2025-05-08|1       |completed   |NULL      |
|ORD0008603|NULL       |EVT0256 |TT00532       |2022-10-30 04:00:00|2022-12-06|2       |completed   |NULL      |
|ORD0012328|NULL       |EVT0118 |TT00240       |2022-06-01 17:00:00|2022-06-16|1       |completed   |NULL      |
|ORD0043240|NULL       |EVT0240 |TT00492       |2024-12

In [12]:
total_orders = orders_cleaned.count()
distinct_orders = orders_cleaned.select("order_id").distinct().count()

print(f"Total Rows: {total_orders}")
print(f"Distinct Order IDs: {distinct_orders}")
print(f"Duplicate Order IDs found: {total_orders - distinct_orders}")

Total Rows: 52520
Distinct Order IDs: 52520
Duplicate Order IDs found: 0


In [13]:
orders_silver = orders_cleaned.fillna({"customer_id": "GUEST"})

orders_silver.select(
    [
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in ["order_id", "customer_id", "quantity", "promo_code"]
    ]
).show()

+--------+-----------+--------+----------+
|order_id|customer_id|quantity|promo_code|
+--------+-----------+--------+----------+
|       0|          0|       0|     29352|
+--------+-----------+--------+----------+



In [14]:
# check quantity for unrealistic number

orders_silver.select("quantity").describe().show()

quantiles = orders_silver.approxQuantile(
    "quantity", [0.25, 0.50, 0.75, 0.95, 0.99, 1.0], 0.01
)
print("Percentiles [25%, 50%, 75%, 95%, 99%, Max]:", quantiles)

+-------+------------------+
|summary|          quantity|
+-------+------------------+
|  count|             52520|
|   mean| 2.547981721249048|
| stddev|1.3551327100983674|
|    min|                 0|
|    max|                 6|
+-------+------------------+

Percentiles [25%, 50%, 75%, 95%, 99%, Max]: [2.0, 2.0, 3.0, 5.0, 6.0, 6.0]


In [15]:
dup_orders = orders_silver.filter(F.col("order_id").contains("_DUP"))

print(f"Total records with '_DUP' in order_id: {dup_orders.count()}")
dup_orders.show(10, truncate=False)

Total records with '_DUP' in order_id: 520
+--------------+----------------+--------+--------------+-------------------+----------+--------+------------+----------+
|order_id      |customer_id     |event_id|ticket_type_id|order_date         |event_date|quantity|order_status|promo_code|
+--------------+----------------+--------+--------------+-------------------+----------+--------+------------+----------+
|ORD0048203_DUP|1e2a435ad7c74203|EVT0160 |TT00325       |2022-03-20 09:00:00|2022-04-02|1       |completed   |NULL      |
|ORD0027374_DUP|e771cb9cbded4054|EVT0076 |TT00155       |2024-03-04 22:00:00|2024-05-31|1       |completed   |NULL      |
|ORD0003043_DUP|bfc8ed144bd74aa8|EVT0240 |TT00492       |2024-07-20 23:00:00|2024-08-17|2       |completed   |SUMMER10  |
|ORD0022897_DUP|c844312a9119439f|EVT0067 |TT00135       |2024-10-03 16:00:00|2024-10-08|1       |completed   |NULL      |
|ORD0021446_DUP|25d94a5b1da34368|EVT0301 |TT00625       |2023-05-29 04:00:00|2023-06-05|6       |comple

In [16]:
orders_silver.filter(
    (F.col("order_id") == "ORD0048203") | (F.col("order_id") == "ORD0048203_DUP")
).show(truncate=False)

+--------------+----------------+--------+--------------+-------------------+----------+--------+------------+----------+
|order_id      |customer_id     |event_id|ticket_type_id|order_date         |event_date|quantity|order_status|promo_code|
+--------------+----------------+--------+--------------+-------------------+----------+--------+------------+----------+
|ORD0048203_DUP|1e2a435ad7c74203|EVT0160 |TT00325       |2022-03-20 09:00:00|2022-04-02|1       |completed   |NULL      |
|ORD0048203    |1e2a435ad7c74203|EVT0160 |TT00325       |2022-03-20 09:00:00|2022-04-02|1       |completed   |NULL      |
+--------------+----------------+--------+--------------+-------------------+----------+--------+------------+----------+



In [17]:
orders_silver = orders_silver.withColumn(
    "order_id", F.regexp_replace(F.col("order_id"), "_DUP", "")
)

orders_silver = orders_silver.dropDuplicates(["order_id"])


In [18]:
orders_silver.groupBy("order_status").count().orderBy(
    F.col("count").desc()
).show(truncate=False)

+------------+-----+
|order_status|count|
+------------+-----+
|completed   |38223|
|cancelled   |8042 |
|confirmed   |3176 |
|no_show     |2559 |
+------------+-----+



In [19]:
orders_silver_path = os.path.join(SILVER, "orders")
orders_silver.write.mode("overwrite").parquet(orders_silver_path)

In [20]:
# Customers
customers_raw = bronze_tables["customers"]

customers_raw.printSchema()

customers_raw.show(5, truncate=False)

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- date_of_birth: string (nullable = true)

+----------------+--------------------------------+----------+---------+-------------------------+-------+------------+--------------+-----------------+-------------+
|customer_id     |customer_unique_id              |first_name|last_name|email                    |city   |country_code|country       |registration_date|date_of_birth|
+----------------+--------------------------------+----------+---------+-------------------------+-------+------------+--------------+-----------------+-------------+
|7ff8bab402524574|4988c52bf1bf47f2b6e3ef3fe0eeb9d3|Simon     |Torre

In [41]:
customers_cleaned = (
     customers_raw
     .withColumn(
        "registration_date",
        F.to_date(
            F.regexp_replace(
                F.regexp_replace(F.col("registration_date"), "/", "-"), # dates are often written with /
                r"^(\d{2})-(\d{2})-(\d{4})$",
                r"$3-$2-$1"
            )
        )
    )
    .withColumn(
        "date_of_birth",
        F.to_date(
            F.regexp_replace(
                F.regexp_replace(F.col("date_of_birth"), "/", "-"),
                r"^(\d{2})-(\d{2})-(\d{4})$",
                r"$3-$2-$1"
            )
        )
    )
)
customers_cleaned.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- country: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- date_of_birth: date (nullable = true)



In [42]:
customers_cleaned.select(
    F.count("*").alias("total_rows"),
    F.count_distinct("customer_id").alias("distinct_customer_ids"),
    F.count_distinct("customer_unique_id").alias("distinct_unique_ids"),
    F.sum(F.when(F.col("customer_id").isNull(), 1).otherwise(0)).alias("null_customer_ids")
).show()

+----------+---------------------+-------------------+-----------------+
|total_rows|distinct_customer_ids|distinct_unique_ids|null_customer_ids|
+----------+---------------------+-------------------+-----------------+
|     10800|                10800|              10692|                0|
+----------+---------------------+-------------------+-----------------+



In [43]:
# Trim whitespace from text fields and lowercase email for consistency
customers_silver = (
    customers_cleaned
    .withColumn("first_name", F.trim(F.col("first_name")))
    .withColumn("last_name", F.trim(F.col("last_name")))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn("city", F.trim(F.col("city")))
    .withColumn("country", F.trim(F.col("country")))
    .withColumn("country_code", F.upper(F.trim(F.col("country_code"))))
)


In [44]:
customers_silver_path = os.path.join(SILVER, "customers")
customers_silver.write.mode("overwrite").parquet(customers_silver_path)

In [45]:
# Events
events_raw = bronze_tables["events"]

events_raw.printSchema()

events_raw.show(5, truncate=False)

root
 |-- event_id: string (nullable = true)
 |-- event_name: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- category: string (nullable = true)
 |-- venue_name: string (nullable = true)
 |-- venue_capacity: string (nullable = true)
 |-- city: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- country: string (nullable = true)
 |-- organizer_since: string (nullable = true)
 |-- is_active: string (nullable = true)

+--------+--------------------+----------+--------+------------------+--------------+---------+------------+-----------+---------------+---------+
|event_id|event_name          |event_type|category|venue_name        |venue_capacity|city     |country_code|country    |organizer_since|is_active|
+--------+--------------------+----------+--------+------------------+--------------+---------+------------+-----------+---------------+---------+
|EVT0001 |Midnight Festival   |Festival  |5       |Innsbruck Grounds |36761         |Innsbruck

In [48]:
events_cleaned = (
    events_raw
    .withColumn("venue_capacity", F.col("venue_capacity").cast(IntegerType()))
    .withColumn("organizer_since", F.to_date(F.col("organizer_since")))
    .withColumn("is_active", F.col("is_active").cast(BooleanType()))
)

events_cleaned.printSchema()

root
 |-- event_id: string (nullable = true)
 |-- event_name: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- category: string (nullable = true)
 |-- venue_name: string (nullable = true)
 |-- venue_capacity: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- country: string (nullable = true)
 |-- organizer_since: date (nullable = true)
 |-- is_active: boolean (nullable = true)



In [49]:
# trim
events_cleaned = (
    events_cleaned
    .withColumn("event_name", F.trim(F.col("event_name")))
    .withColumn("event_type", F.trim(F.col("event_type")))
    .withColumn("venue_name", F.trim(F.col("venue_name")))
    .withColumn("city", F.trim(F.col("city")))
    .withColumn("country", F.trim(F.col("country")))
    .withColumn("country_code", F.upper(F.trim(F.col("country_code"))))
)

events_cleaned.show(5, truncate=False)

+--------+--------------------+----------+--------+------------------+--------------+---------+------------+-----------+---------------+---------+
|event_id|event_name          |event_type|category|venue_name        |venue_capacity|city     |country_code|country    |organizer_since|is_active|
+--------+--------------------+----------+--------+------------------+--------------+---------+------------+-----------+---------------+---------+
|EVT0001 |Midnight Festival   |Festival  |5       |Innsbruck Grounds |36761         |Innsbruck|AT          |Austria    |2019-10-14     |true     |
|EVT0002 |Crystal Drama Series|Theatre   |2       |Eindhoven Pavilion|618           |Eindhoven|NL          |Netherlands|2020-05-08     |true     |
|EVT0003 |Vintage Open Air    |Festival  |4       |Lyon Arena        |6394          |Lyon     |FR          |France     |2015-09-23     |true     |
|EVT0004 |Summer Playhouse    |Theather  |4       |Lille Grounds     |12351         |Lille    |FR          |France    

In [51]:
# check duplicate id
total_events = events_cleaned.count()
distinct_events = events_cleaned.select("event_id").distinct().count()

print(f"Duplicates: {total_events - distinct_events}")


Duplicates: 0


In [53]:
# save
events_silver = events_cleaned
events_silver_path = os.path.join(SILVER, "events")
events_silver.write.mode("overwrite").parquet(events_silver_path)




In [54]:
# ticket_types
ticket_types_raw = bronze_tables["ticket_types"]

ticket_types_raw.printSchema()

ticket_types_raw.show(5, truncate=False)

root
 |-- ticket_type_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- ticket_class: string (nullable = true)
 |-- price: string (nullable = true)
 |-- max_per_order: string (nullable = true)
 |-- merch_included: string (nullable = true)
 |-- refund_policy: string (nullable = true)

+--------------+--------+-----------------+------+-------------+--------------+-------------+
|ticket_type_id|event_id|ticket_class     |price |max_per_order|merch_included|refund_policy|
+--------------+--------+-----------------+------+-------------+--------------+-------------+
|TT00001       |EVT0001 |General Admission|178.06|4            |False         |moderate     |
|TT00002       |EVT0001 |Seated           |198.17|4            |False         |flexible     |
|TT00003       |EVT0001 |Balcony          |215.36|10           |False         |moderate     |
|TT00004       |EVT0002 |General Admission|39.43 |4            |False         |moderate     |
|TT00005       |EVT0002 |Seated  

In [57]:
ticket_types_cleaned = (
    ticket_types_raw
    .withColumn("price", F.col("price").cast(DecimalType(10, 2)))
    .withColumn("max_per_order", F.col("max_per_order").cast(IntegerType()))
    .withColumn("merch_included", F.col("merch_included").cast(BooleanType()))
)

ticket_types_cleaned.printSchema()

root
 |-- ticket_type_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- ticket_class: string (nullable = true)
 |-- price: decimal(10,2) (nullable = true)
 |-- max_per_order: integer (nullable = true)
 |-- merch_included: boolean (nullable = true)
 |-- refund_policy: string (nullable = true)



In [65]:
# trim
ticket_types_cleaned = (
    ticket_types_cleaned
    .withColumn("ticket_class", F.trim(F.col("ticket_class")))
    .withColumn("refund_policy", F.trim(F.col("refund_policy")))
)

ticket_types_cleaned.show(5, truncate=False)

+--------------+--------+-----------------+------+-------------+--------------+-------------+
|ticket_type_id|event_id|ticket_class     |price |max_per_order|merch_included|refund_policy|
+--------------+--------+-----------------+------+-------------+--------------+-------------+
|TT00001       |EVT0001 |General Admission|178.06|4            |false         |moderate     |
|TT00002       |EVT0001 |Seated           |198.17|4            |false         |flexible     |
|TT00003       |EVT0001 |Balcony          |215.36|10           |false         |moderate     |
|TT00004       |EVT0002 |General Admission|39.43 |4            |false         |moderate     |
|TT00005       |EVT0002 |Seated           |43.30 |8            |false         |moderate     |
+--------------+--------+-----------------+------+-------------+--------------+-------------+
only showing top 5 rows


In [67]:
total_tickets = ticket_types_cleaned.count()
distinct_tickets = ticket_types_cleaned.select("ticket_type_id").distinct().count()

print(f"Duplicates: {total_tickets - distinct_tickets}")

Duplicates: 0


In [70]:
#save
ticket_types_silver = ticket_types_cleaned
ticket_types_silver_path = os.path.join(SILVER, "ticket_types")
ticket_types_silver.write.mode("overwrite").parquet(ticket_types_silver_path)


In [73]:
# Payments
payments_raw = bronze_tables["payments"]

payments_raw.printSchema()

payments_raw.show(5, truncate=False)

root
 |-- payment_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- payment_date: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- currency: string (nullable = true)

+-----------+----------+-------------------+------+--------------+--------------+--------+
|payment_id |order_id  |payment_date       |amount|payment_method|payment_status|currency|
+-----------+----------+-------------------+------+--------------+--------------+--------+
|PAY00041147|ORD0049816|2022-09-30 03:00:00|25.53 |credit_card   |completed     |EUR     |
|PAY00003085|ORD0050207|2024-11-15 00:00:00|608.17|credit_card   |completed     |EUR     |
|PAY00020943|ORD0032796|2024-02-23 05:00:00|96.97 |credit_card   |completed     |EUR     |
|PAY00002597|ORD0048065|2024-08-20 22:00:00|171.01|credit_card   |completed     |EUR     |
|PAY00022163|ORD0049181|2022-12-12 14:00:00|855.68|credit_car

In [74]:
payments_cleaned = (
    payments_raw
    .withColumn("payment_date", F.to_timestamp(F.col("payment_date")))
    .withColumn("amount", F.col("amount").cast(DecimalType(10, 2)))
)

In [75]:
payments_cleaned = (
    payments_cleaned
    .withColumn("currency", F.upper(F.trim(F.col("currency"))))
    .withColumn("payment_method", F.lower(F.trim(F.col("payment_method"))))
    .withColumn("payment_status", F.lower(F.trim(F.col("payment_status"))))
    .withColumn("order_id", F.trim(F.col("order_id")))
)

payments_cleaned.show(5, truncate=False)

+-----------+----------+-------------------+------+--------------+--------------+--------+
|payment_id |order_id  |payment_date       |amount|payment_method|payment_status|currency|
+-----------+----------+-------------------+------+--------------+--------------+--------+
|PAY00041147|ORD0049816|2022-09-30 03:00:00|25.53 |credit_card   |completed     |EUR     |
|PAY00003085|ORD0050207|2024-11-15 00:00:00|608.17|credit_card   |completed     |EUR     |
|PAY00020943|ORD0032796|2024-02-23 05:00:00|96.97 |credit_card   |completed     |EUR     |
|PAY00002597|ORD0048065|2024-08-20 22:00:00|171.01|credit_card   |completed     |EUR     |
|PAY00022163|ORD0049181|2022-12-12 14:00:00|855.68|credit_card   |completed     |EUR     |
+-----------+----------+-------------------+------+--------------+--------------+--------+
only showing top 5 rows


In [76]:
total_payments = payments_cleaned.count()
distinct_payments = payments_cleaned.select("payment_id").distinct().count()

print(f"Duplicates: {total_payments - distinct_payments}")

Duplicates: 0


In [79]:
#save
payments_silver = payments_cleaned
payments_silver_path = os.path.join(SILVER, "payments")
payments_silver.write.mode("overwrite").parquet(payments_silver_path)

In [80]:
#event_categories
event_categories_raw = bronze_tables["event_categories"]

event_categories_raw.printSchema()

event_categories_raw.show(10, truncate=False)

root
 |-- category_code: string (nullable = true)
 |-- category_name: string (nullable = true)
 |-- category_description: string (nullable = true)
 |-- price_range: string (nullable = true)

+-------------+-------------+------------------------------------------------+-----------+
|category_code|category_name|category_description                            |price_range|
+-------------+-------------+------------------------------------------------+-----------+
|1            |Local        |Small local events, community venues            |€0–€25     |
|2            |Club         |Club shows and small halls, up to 1,500 visitors|€25–€50    |
|3            |Standard     |Mid-size events in established venues           |€50–€90    |
|4            |Major        |Large events, arena and stadium acts            |€90–€160   |
|5            |Flagship     |Flagship festivals and international headliners |€160+      |
+-------------+-------------+------------------------------------------------+---

In [82]:
event_categories_silver = (
    event_categories_raw
    .withColumn("category_code", F.col("category_code").cast(IntegerType()))
    .withColumn("category_name", F.trim(F.col("category_name")))
    .withColumn("category_description", F.trim(F.col("category_description")))
    .withColumn("price_range", F.trim(F.col("price_range")))
)

event_categories_path = os.path.join(SILVER, "event_categories")
event_categories_silver.write.mode("overwrite").parquet(event_categories_path)


In [83]:
# Reviews
reviews_raw = bronze_tables["reviews"]

reviews_raw.printSchema()

reviews_raw.show(5, truncate=False)

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- venue_score: string (nullable = true)
 |-- organisation_score: string (nullable = true)
 |-- value_score: string (nullable = true)
 |-- review_date: string (nullable = true)
 |-- review_text: string (nullable = true)

+----------+----------+----------------+--------+------+-----------+------------------+-----------+-----------+---------------------------------------+
|review_id |order_id  |customer_id     |event_id|rating|venue_score|organisation_score|value_score|review_date|review_text                            |
+----------+----------+----------------+--------+------+-----------+------------------+-----------+-----------+---------------------------------------+
|REV0000001|ORD0029337|16250810a90947d2|EVT0142 |6.7   |7.2        |10.0              |7.0        |2023-12-26 |Que

In [84]:
reviews_cleaned = (
    reviews_raw
    .withColumn("rating", F.col("rating").cast(DecimalType(3, 1)))
    .withColumn("venue_score", F.col("venue_score").cast(DecimalType(3, 1)))
    .withColumn("organisation_score", F.col("organisation_score").cast(DecimalType(3, 1)))
    .withColumn("value_score", F.col("value_score").cast(DecimalType(3, 1)))
    .withColumn("review_date", F.to_date(F.col("review_date")))
)

reviews_cleaned.printSchema()

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- rating: decimal(3,1) (nullable = true)
 |-- venue_score: decimal(3,1) (nullable = true)
 |-- organisation_score: decimal(3,1) (nullable = true)
 |-- value_score: decimal(3,1) (nullable = true)
 |-- review_date: date (nullable = true)
 |-- review_text: string (nullable = true)



In [85]:
# trim
reviews_cleaned = (
    reviews_cleaned
    .withColumn("review_id", F.trim(F.col("review_id")))
    .withColumn("order_id", F.trim(F.col("order_id")))
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("event_id", F.trim(F.col("event_id")))
    .withColumn("review_text", F.trim(F.col("review_text")))
)

reviews_cleaned.show(5, truncate=False)

+----------+----------+----------------+--------+------+-----------+------------------+-----------+-----------+---------------------------------------+
|review_id |order_id  |customer_id     |event_id|rating|venue_score|organisation_score|value_score|review_date|review_text                            |
+----------+----------+----------------+--------+------+-----------+------------------+-----------+-----------+---------------------------------------+
|REV0000001|ORD0029337|16250810a90947d2|EVT0142 |6.7   |7.2        |10.0              |7.0        |2023-12-26 |Queue at the entrance was way too long.|
|REV0000002|ORD0034769|c818549e1d8341d2|EVT0433 |8.7   |10.0       |7.3               |4.6        |2021-07-18 |NULL                                   |
|REV0000003|ORD0046313|f17d4ba7c53d47cc|EVT0412 |8.6   |9.3        |5.1               |5.9        |2023-03-14 |Organisation could be better.          |
|REV0000004|ORD0032998|39ed28703d4b46d7|EVT0078 |9.4   |5.1        |6.9               |5

In [86]:
total_reviews = reviews_cleaned.count()
distinct_reviews = reviews_cleaned.select("review_id").distinct().count()

print(f"Duplicates: {total_reviews - distinct_reviews}")

Duplicates: 0


In [88]:
#save
reviews_silver = reviews_cleaned
reviews_silver_path = os.path.join(SILVER, "reviews")
reviews_silver.write.mode("overwrite").parquet(reviews_silver_path)

